# 7주차 실습: 이진 탐색 트리(BST) 구현 및 성능 분석 — 학생용

## 목표·규약
재귀 탐색/삽입/삭제, 중위 정렬 검증, 삽입 순서별 높이·탐색 시간 비교.
Python 3.8+; 위에서 아래로 실행. 키는 비교 가능한 정수, 중복 삽입 무시, 없는 값 삭제는 변화 없음.
높이는 간선 기준(빈 트리 -1). 단순 BST는 균형을 보장하지 않습니다.
학생 TODO는 NotImplementedError로 즉시 중단하며 구현 후 테스트합니다.


In [1]:
import random, time, statistics
from collections import deque
print("환경 준비 완료")

환경 준비 완료


## Step 1. BSTNode
지난주 TreeNode의 data/left/right 구조를 재사용.

In [2]:
class BSTNode:
    def __init__(self, data):
        self.data = data
        self.left = None
        self.right = None

    def __repr__(self):
        return f"BSTNode({self.data!r})"

## Step 2–3. 전체 BST 클래스
재귀 결과를 `node.left/right`, 최상위 결과를 `self.root`에 재연결하세요.
삭제: 리프→None / 자식 하나→자식 / 둘→오른쪽 최소값 복사 후 후속자 삭제.
후속자는 오른쪽 자식을 가질 수 있어 재연결이 필요합니다.
하나의 클래스에 전체 메서드를 넣어 동일 이름 자기 상속을 피했습니다.
성능 실험용 반복 메서드도 포함합니다. 큰 편향 트리에 재귀 메서드를 호출하지 마세요.


In [3]:
class BinarySearchTree:
    def __init__(self):
        self.root = None

    # ---------- 재귀 구현 ----------
    def search(self, key):
        return self._search(self.root, key)

    def _search(self, node, key):
        if node is None:
            return False
        if key == node.data:
            return True
        if key < node.data:
            return self._search(node.left, key)
        return self._search(node.right, key)

    def insert(self, key):
        self.root = self._insert(self.root, key)

    def _insert(self, node, key):
        if node is None:
            return BSTNode(key)
        if key < node.data:
            node.left = self._insert(node.left, key)
        elif key > node.data:
            node.right = self._insert(node.right, key)
        # key == node.data 이면 중복이므로 아무것도 하지 않는다
        return node

    def delete(self, key):
        self.root = self._delete(self.root, key)

    def _delete(self, node, key):
        if node is None:
            return None  # 없는 값이면 트리는 그대로
        if key < node.data:
            node.left = self._delete(node.left, key)
        elif key > node.data:
            node.right = self._delete(node.right, key)
        else:
            # 자식이 0개 또는 1개: 있는 쪽(없으면 None)으로 대체
            if node.left is None:
                return node.right
            if node.right is None:
                return node.left
            # 자식이 2개: 오른쪽 서브트리의 최솟값(후속자)을 복사한 뒤 그 노드를 삭제
            successor = self._find_min(node.right)
            node.data = successor.data
            node.right = self._delete(node.right, successor.data)
        return node

    def _find_min(self, node):
        while node.left is not None:
            node = node.left
        return node

    def inorder(self):
        result = []

        def visit(node):
            if node is None:
                return
            visit(node.left)
            result.append(node.data)
            visit(node.right)

        visit(self.root)
        return result

    def height(self):
        def calc(node):
            if node is None:
                return -1
            return 1 + max(calc(node.left), calc(node.right))

        return calc(self.root)

    def count_nodes(self):
        def calc(node):
            if node is None:
                return 0
            return 1 + calc(node.left) + calc(node.right)

        return calc(self.root)

    # ---------- 반복 구현 (성능 실험용) ----------
    def insert_iterative(self, key):
        if self.root is None:
            self.root = BSTNode(key)
            return
        cur = self.root
        while True:
            if key == cur.data:
                return  # 중복
            if key < cur.data:
                if cur.left is None:
                    cur.left = BSTNode(key)
                    return
                cur = cur.left
            else:
                if cur.right is None:
                    cur.right = BSTNode(key)
                    return
                cur = cur.right

    def search_iterative(self, key):
        cur = self.root
        while cur is not None:
            if key == cur.data:
                return True
            cur = cur.left if key < cur.data else cur.right
        return False

    def height_iterative(self):
        if self.root is None:
            return -1
        best = 0
        stack = [(self.root, 0)]
        while stack:
            node, depth = stack.pop()
            if depth > best:
                best = depth
            if node.left is not None:
                stack.append((node.left, depth + 1))
            if node.right is not None:
                stack.append((node.right, depth + 1))
        return best


## Step 4. 삽입/삭제 로직 검증
후속자의 오른쪽 자식, 루트 삭제, 중복, 없는 값, 무작위 혼합 연산까지 검사.

In [4]:
def make_bst(values, iterative=False):
    tree = BinarySearchTree()
    for value in values:
        if iterative: tree.insert_iterative(value)
        else: tree.insert(value)
    return tree

def assert_bst(tree, expected):
    # 조상 범위까지 검사: 바로 위 부모만 비교하는 오류 방지
    def valid(node, low=None, high=None):
        if node is None: return True
        if low is not None and node.data <= low: return False
        if high is not None and node.data >= high: return False
        return valid(node.left, low, node.data) and valid(node.right, node.data, high)
    assert valid(tree.root)
    assert tree.inorder() == sorted(set(expected))
    assert tree.count_nodes() == len(set(expected))


In [5]:
values = [50, 30, 70, 20, 40, 60, 80]
bst = make_bst(values)
assert_bst(bst, values)
assert bst.search(40) is True and bst.search(99) is False
for key in [20, 30, 70]:
    bst.delete(key)
    values.remove(key)
    assert_bst(bst, values)
    assert not bst.search(key)
    print("삭제", key, bst.inorder())
# 루트 두 자식 삭제
bst.delete(50); values.remove(50)
assert_bst(bst, values)
assert bst.root.data == 60

empty = BinarySearchTree()
empty.delete(1)
assert_bst(empty, [])
assert empty.height() == -1 and not empty.search(1)
for vals in [[5], [5,3], [5,7]]:
    t = make_bst(vals)
    t.delete(5)
    assert_bst(t, vals[1:])
# 후속자 60은 오른쪽 자식65를 갖는다: 65가 손실되면 안 됨.
t = make_bst([50,30,70,60,65,80])
t.delete(50)
assert t.root.data == 60 and t.root.right.left.data == 65
assert_bst(t, [30,60,65,70,80])
t.insert(65); t.delete(999)
assert_bst(t, [30,60,65,70,80])
# 전체 삭제로 빈 트리 복귀
for key in [30,60,65,70,80]: t.delete(key)
assert_bst(t, [])
# 고정 시드의 혼합 삽입/삭제를 Python set과 대조
rng = random.Random(7)
t, reference = BinarySearchTree(), set()
for _ in range(200):
    key = rng.randrange(50)
    if rng.random() < 0.6: t.insert(key); reference.add(key)
    else: t.delete(key); reference.discard(key)
    assert_bst(t, reference)
    assert t.search(key) == (key in reference)
print("모든 삽입·삭제·경계·불변식 테스트 통과")


삭제 20 [30, 40, 50, 60, 70, 80]
삭제 30 [40, 50, 60, 70, 80]
삭제 70 [40, 50, 60, 80]
모든 삽입·삭제·경계·불변식 테스트 통과


## Step 5. 높이 비교 실험
n=1000 재귀 삽입은 편향 트리에서 RecursionError가 날 수 있습니다. 반복 구현을 이용하며 재귀 제한을 올리지 않습니다.
최소 가능 높이 n>=1: ceil(log2(n+1))-1 = n.bit_length()-1.
무작위 트리는 기대 높이 O(log n)이지만 매번 균형이 보장되는 것은 아닙니다.


In [6]:
def build_for_experiment(n, order="random", seed=7):
    keys = list(range(n))
    if order == "random":
        random.Random(seed).shuffle(keys)   # 고정 시드로 재현 가능하게
    elif order != "sorted":
        raise ValueError("order는 'random' 또는 'sorted'만 가능하다")
    tree = BinarySearchTree()
    for key in keys:
        tree.insert_iterative(key)          # 재귀 깊이 제한을 피하려고 반복 삽입 사용
    return tree

def measure_height(n, order="random", seed=7):
    return build_for_experiment(n, order, seed).height_iterative()

n = 1000
bst_random = build_for_experiment(n, "random")
bst_sorted = build_for_experiment(n, "sorted")
assert bst_sorted.height_iterative() == n-1
print("최소 가능 높이:", n.bit_length()-1)
print("무작위/정렬 높이:", bst_random.height_iterative(), bst_sorted.height_iterative())


최소 가능 높이: 9
무작위/정렬 높이: 24 999


## Step 6. 탐색 시간 측정
동일한 키 목록, 반복 탐색 구현, 구성 시간 제외, 워밍업, 5회 중앙값. 측정값은 환경에 따라 달라집니다.

In [7]:
targets = [random.Random(17+i).randrange(n) for i in range(1000)]
def timed_search(tree, targets, repeats=5):
    # 워밍업: 측정에 포함하지 않는 1회 실행
    for key in targets:
        tree.search_iterative(key)

    elapsed = []
    for _ in range(repeats):
        start = time.perf_counter()
        for key in targets:
            tree.search_iterative(key)
        elapsed.append(time.perf_counter() - start)
    return statistics.median(elapsed)
random_time = timed_search(bst_random, targets)
sorted_time = timed_search(bst_sorted, targets)
print(f"무작위 트리 탐색 중앙값: {random_time:.6f}s")
print(f"정렬 삽입 트리 중앙값: {sorted_time:.6f}s")
if random_time > 0: print(f"측정 시간비: {sorted_time/random_time:.2f}")

무작위 트리 탐색 중앙값: 0.000670s
정렬 삽입 트리 중앙값: 0.020330s
측정 시간비: 30.32


## 과제: BST 구현 및 성능 분석
1. 모든 로직 테스트 통과. 2. n=100/1000/10000의 두 삽입 순서 높이를 표로 정리.
3. 최소 높이와 비교하고 성능 영향을 3~5문장으로 분석. 시간 측정 조건도 명시.
4. 개인 GitHub Week7 폴더, 주당 최소 2회 커밋, 과제 폼 GitHub 링크 제출.
**주의:** 정렬 순서 실제 삽입은 전체 O(n²)입니다. n=10000 셀은 느릴 수 있어 마지막에 실행합니다. 체인 직접 구성은 같은 삽입 실험이 아니므로 대체하지 않습니다.


In [8]:
rows = []
for size in [100, 1000, 10000]:
    h_random = measure_height(size, "random")
    h_sorted = measure_height(size, "sorted")
    assert h_sorted == size - 1
    rows.append((size, size.bit_length()-1, h_random, h_sorted))
print("n | 최소 가능 높이 | 무작위 높이 | 정렬 높이")
for row in rows: print(*row, sep=" | ")


n | 최소 가능 높이 | 무작위 높이 | 정렬 높이
100 | 6 | 12 | 99
1000 | 9 | 24 | 999
10000 | 13 | 29 | 9999


## 분석 작성란
**높이 비교표 (간선 기준, 무작위는 seed=7)**

| n | 최소 가능 높이 | 무작위 삽입 높이 | 정렬 삽입 높이 |
|---|---|---|---|
| 100 | 6 | 12 | 99 |
| 1000 | 9 | 24 | 999 |
| 10000 | 13 | 29 | 9999 |

- 무작위/정렬 입력에서 높이: 무작위 삽입은 n=100, 1000, 10000에서 각각 12, 24, 29로 n이 100배 커지는 동안 높이는 약 2.4배만 늘었고, 정렬 삽입은 99, 999, 9999로 항상 n-1이었다.
- 동일 탐색 대상의 시간 중앙값과 실험 조건: n=1000 트리에서 같은 탐색 대상 1000개(시드 고정)를 반복 탐색 구현으로 찾았고, 트리 구성 시간은 제외하고 워밍업 1회 뒤 perf_counter로 5회 측정한 중앙값을 썼으며, 무작위 트리는 0.000670초, 정렬 삽입 트리는 0.020330초로 약 30배 차이가 났다. 절대 시간은 실행 환경에 따라 달라지므로 비율을 중심으로 보아야 한다고 생각한다.
- 편향 원인과 균형 보장 여부: 정렬된 키를 차례로 넣으면 새 키가 항상 현재 최댓값보다 커서 오른쪽 자식으로만 이어지므로 연결 리스트와 같은 모양이 되고, 단순 BST는 삽입 순서에 따라 모양이 결정될 뿐 균형을 보장하지 않는다.
- 최소 높이와 무작위 높이가 다른 이유: 최소 높이는 모든 레벨을 빈틈없이 채운 이상적인 경우의 값이지만, 무작위 삽입에서는 먼저 들어온 키가 루트와 윗부분 구조를 정하므로 한쪽으로 조금씩 치우쳐 최소 높이보다 커진다고 생각한다. 그래도 O(log n) 수준에 머물러 있어 정렬 삽입의 O(n)과는 큰 차이가 난다.

## 복잡도 정리
search/insert/delete O(h+1), 재귀 보조 공간 O(h+1). 반복 탐색/삽입 공간 O(1).
중위 순회 O(n) 시간, O(h+1) 호출 스택 + O(n) 출력. 높이 계산 O(n).
균형 높이 O(log n), 최악 O(n); 서로 다른 키의 무작위 순열 삽입은 기대 O(log n) 연산.
Tree Sort 기대 O(n log n), 최악 O(n²); 중복을 보존하려면 카운터 등 정책 변경 필요.


## 선택 심화·시각화
- VisuAlgo BST 모드에서 50,30,70,20,40,60,80 삽입 후 20,30,70 삭제와 코드 결과 비교.
- 중위 선행자를 이용한 삭제로 변경.
- 셔플 전처리는 기대 성능 개선이지 최악 균형 보장이 아님을 설명.
- 자손 수 저장으로 kth_smallest 구현: O(h), 균형일 때만 O(log n). 삽입/삭제마다 size 갱신 필요.
- 다음 주 중간 평가: 연결리스트, 스택/큐, 트리, BST 복습.


## 선택 심화 1. VisuAlgo 삭제 비교 (50, 30, 70, 20, 40, 60, 80 삽입 후 20, 30, 70 삭제)
VisuAlgo BST 모드에서 같은 순서로 삽입하고 20 → 30 → 70을 차례로 삭제한 화면과 아래 코드 출력의 구조를 비교한다. 아래 코드는 삭제할 때마다 트리를 레벨별로 출력한다.

In [9]:
def levels(tree):
    # 레벨별 노드 값 목록 (구조를 눈으로 비교하기 위한 용도)
    result = []
    if tree.root is None:
        return result
    queue = deque([tree.root])
    while queue:
        level = []
        for _ in range(len(queue)):
            node = queue.popleft()
            level.append(node.data)
            if node.left is not None:
                queue.append(node.left)
            if node.right is not None:
                queue.append(node.right)
        result.append(level)
    return result

demo = make_bst([50, 30, 70, 20, 40, 60, 80])
print("삽입 직후     :", levels(demo))
expected_levels = {
    20: [[50], [30, 70], [40, 60, 80]],
    30: [[50], [40, 70], [60, 80]],
    70: [[50], [40, 80], [60]],
}
for key in [20, 30, 70]:
    demo.delete(key)
    print(f"{key} 삭제 후     :", levels(demo), "| 중위:", demo.inorder())
    assert levels(demo) == expected_levels[key]

삽입 직후     : [[50], [30, 70], [20, 40, 60, 80]]
20 삭제 후     : [[50], [30, 70], [40, 60, 80]] | 중위: [30, 40, 50, 60, 70, 80]
30 삭제 후     : [[50], [40, 70], [60, 80]] | 중위: [40, 50, 60, 70, 80]
70 삭제 후     : [[50], [40, 80], [60]] | 중위: [40, 50, 60, 80]


**비교 결과 정리**
- 20 삭제는 리프 삭제라서 30의 왼쪽 자리만 비게 된다.
- 30 삭제는 자식이 하나(40)뿐인 경우라서 40이 30의 자리를 그대로 대신한다.
- 70 삭제는 자식이 둘인 경우라서 오른쪽 서브트리의 최솟값 80을 70 자리에 복사한 뒤 원래의 80 노드를 지운다. 그 결과 루트 50 아래 40, 80이 오고 80의 왼쪽에 60이 남는다.
- VisuAlgo 화면과 구조가 같다면 삭제 로직이 올바르다고 판단하고, 다르다면 두 자식 삭제 시 후속자를 쓰는지 선행자를 쓰는지부터 확인해야 한다고 생각한다.

## 선택 심화 2. 중위 선행자를 이용한 삭제
자식이 둘인 노드를 지울 때 오른쪽 최솟값(후속자) 대신 **왼쪽 서브트리의 최댓값(중위 선행자)** 을 복사하도록 바꾼다. 기존 테스트(루트 삭제 시 60)는 후속자 방식을 전제로 하므로, 원래 클래스는 그대로 두고 상속해서 `_delete`만 바꾼다.

In [10]:
class BSTPredecessor(BinarySearchTree):
    def _find_max(self, node):
        while node.right is not None:
            node = node.right
        return node

    def _delete(self, node, key):
        if node is None:
            return None
        if key < node.data:
            node.left = self._delete(node.left, key)
        elif key > node.data:
            node.right = self._delete(node.right, key)
        else:
            if node.left is None:
                return node.right
            if node.right is None:
                return node.left
            # 선행자: 왼쪽 서브트리의 최댓값. 선행자는 왼쪽 자식을 가질 수 있어 재연결이 필요하다
            predecessor = self._find_max(node.left)
            node.data = predecessor.data
            node.left = self._delete(node.left, predecessor.data)
        return node


def make_pred(values):
    tree = BSTPredecessor()
    for value in values:
        tree.insert(value)
    return tree

# 루트 삭제: 선행자 40이 올라온다
t = make_pred([50, 30, 70, 20, 40, 60, 80])
t.delete(50)
assert t.root.data == 40
assert_bst(t, [20, 30, 40, 60, 70, 80])

# 선행자 40이 왼쪽 자식 35를 가진 경우, 35가 사라지면 안 된다
t = make_pred([50, 30, 70, 20, 40, 35, 80])
t.delete(50)
assert t.root.data == 40 and t.root.left.right.data == 35
assert_bst(t, [20, 30, 35, 40, 70, 80])

# 없는 값 / 빈 트리
t.delete(999)
assert_bst(t, [20, 30, 35, 40, 70, 80])
e = BSTPredecessor()
e.delete(1)
assert_bst(e, [])

# 고정 시드 혼합 연산을 Python set과 대조 (후속자 버전과 같은 방식)
rng = random.Random(11)
t, reference = BSTPredecessor(), set()
for _ in range(300):
    key = rng.randrange(60)
    if rng.random() < 0.55:
        t.insert(key)
        reference.add(key)
    else:
        t.delete(key)
        reference.discard(key)
    assert_bst(t, reference)
    assert t.search(key) == (key in reference)
print("선행자 삭제 테스트 통과")

선행자 삭제 테스트 통과


## 선택 심화 3. 셔플 전처리의 의미
정렬된 키를 섞은 뒤 삽입하면 편향 트리를 피할 수 있다. 다만 이것은 **기대 성능을 좋게 만드는 방법일 뿐 최악의 경우의 균형을 보장하지는 않는다.** 아래에서 같은 키 1000개를 시드만 바꿔 30번 섞어 삽입한 높이를 확인한다.

In [11]:
n = 1000
heights = sorted(measure_height(n, "random", seed) for seed in range(30))
print("최소 가능 높이        :", n.bit_length() - 1)
print("셔플 30회 높이 (최소/중앙/최대):", heights[0], heights[len(heights) // 2], heights[-1])
print("정렬 그대로 삽입 높이  :", measure_height(n, "sorted"))
assert heights[-1] < n // 10

최소 가능 높이        : 9
셔플 30회 높이 (최소/중앙/최대): 19 21 25
정렬 그대로 삽입 높이  : 999


**설명**
- 서로 다른 키를 무작위 순열로 삽입하면 트리 높이의 기대값이 O(log n)이 된다. 실험에서도 30번 모두 정렬 삽입(999)보다 훨씬 낮은 높이가 나왔다.
- 하지만 셔플 결과가 우연히 오름차순에 가까운 순열이 될 가능성이 0은 아니다. 그런 경우 높이는 O(n)까지 커질 수 있으므로 균형이 보장된다고 말할 수는 없다고 생각한다. 다만 n이 크면 그런 순열이 나올 확률은 거의 없다.
- 매번 최악의 경우에도 O(log n)을 보장하려면 AVL, 레드-블랙 트리처럼 삽입·삭제 때 스스로 균형을 맞추는 자료구조가 필요하다.

## 선택 심화 4. 자손 수 저장으로 kth_smallest 구현
각 노드에 자기 서브트리의 노드 수(`size`)를 저장한다. k번째로 작은 값을 찾을 때, 왼쪽 서브트리 크기 `L`과 비교해서 `k == L + 1`이면 현재 노드, `k <= L`이면 왼쪽, 그 외에는 `k - (L + 1)`번째를 오른쪽에서 찾는다. 한 번에 한 경로만 내려가므로 시간은 O(h)이다. 삽입과 삭제 때마다 경로에 있는 노드의 `size`를 갱신해야 한다.

In [12]:
class SizedNode(BSTNode):
    def __init__(self, data):
        super().__init__(data)
        self.size = 1


class OrderStatisticBST(BinarySearchTree):
    @staticmethod
    def _size(node):
        return node.size if node is not None else 0

    def _update(self, node):
        node.size = 1 + self._size(node.left) + self._size(node.right)

    def _insert(self, node, key):
        if node is None:
            return SizedNode(key)
        if key < node.data:
            node.left = self._insert(node.left, key)
        elif key > node.data:
            node.right = self._insert(node.right, key)
        else:
            return node          # 중복이면 크기 변화 없음
        self._update(node)       # 돌아오는 길에 경로의 size 갱신
        return node

    def _delete(self, node, key):
        if node is None:
            return None
        if key < node.data:
            node.left = self._delete(node.left, key)
        elif key > node.data:
            node.right = self._delete(node.right, key)
        else:
            if node.left is None:
                return node.right
            if node.right is None:
                return node.left
            successor = self._find_min(node.right)
            node.data = successor.data
            node.right = self._delete(node.right, successor.data)
        self._update(node)
        return node

    def insert_iterative(self, key):
        # 부모 클래스의 반복 삽입은 size를 갱신하지 않으므로 이 클래스에서는 재정의한다
        if self.search_iterative(key):
            return
        if self.root is None:
            self.root = SizedNode(key)
            return
        cur = self.root
        while True:
            cur.size += 1
            if key < cur.data:
                if cur.left is None:
                    cur.left = SizedNode(key)
                    return
                cur = cur.left
            else:
                if cur.right is None:
                    cur.right = SizedNode(key)
                    return
                cur = cur.right

    def kth_smallest(self, k):
        if k < 1 or k > self._size(self.root):
            raise IndexError("k가 범위를 벗어났다")
        node = self.root
        while node is not None:
            left_size = self._size(node.left)
            if k == left_size + 1:
                return node.data
            if k <= left_size:
                node = node.left
            else:
                k -= left_size + 1
                node = node.right


def check_sizes(node):
    # 모든 노드의 size가 실제 서브트리 크기와 같은지 검사
    if node is None:
        return 0
    total = 1 + check_sizes(node.left) + check_sizes(node.right)
    assert node.size == total
    return total


t = OrderStatisticBST()
for v in [50, 30, 70, 20, 40, 60, 80]:
    t.insert(v)
print("1~7번째 작은 값:", [t.kth_smallest(i) for i in range(1, 8)])
assert [t.kth_smallest(i) for i in range(1, 8)] == [20, 30, 40, 50, 60, 70, 80]
for bad in [0, 8]:
    try:
        t.kth_smallest(bad)
        raise AssertionError("범위 밖 k는 예외가 나야 한다")
    except IndexError:
        pass

# 고정 시드 혼합 연산: 매 단계마다 size와 모든 k번째 값을 정렬 리스트와 대조
rng = random.Random(5)
t, reference = OrderStatisticBST(), set()
for _ in range(300):
    key = rng.randrange(60)
    if rng.random() < 0.6:
        t.insert(key)
        reference.add(key)
    else:
        t.delete(key)
        reference.discard(key)
    check_sizes(t.root)
    ordered = sorted(reference)
    assert [t.kth_smallest(i) for i in range(1, len(ordered) + 1)] == ordered

# 반복 삽입 경로도 size가 맞는지 확인
t2 = OrderStatisticBST()
for v in [8, 3, 10, 1, 6, 14, 4, 7, 13, 6]:
    t2.insert_iterative(v)
check_sizes(t2.root)
assert [t2.kth_smallest(i) for i in range(1, t2.count_nodes() + 1)] == [1, 3, 4, 6, 7, 8, 10, 13, 14]
print("kth_smallest 테스트 통과")

1~7번째 작은 값: [20, 30, 40, 50, 60, 70, 80]
kth_smallest 테스트 통과


**복잡도**
- `kth_smallest`는 루트에서 한 경로만 내려가므로 O(h)이다. 트리가 균형일 때만 O(log n)이고, 편향 트리에서는 O(n)이다.
- 그냥 중위 순회로 k번째를 찾으면 O(n)이므로, 같은 트리에서 여러 번 질의할 때는 size를 저장하는 방식이 유리하다고 생각한다.
- 대신 삽입과 삭제마다 경로의 `size`를 갱신해야 해서 구현이 복잡해지고 저장 공간이 노드당 정수 하나씩 늘어난다.

## 선택 심화 5. 다음 주 중간 평가 복습 정리
- **연결리스트**: 삽입·삭제는 위치를 알면 O(1)이지만 탐색은 O(n)이고, 임의 접근이 불가능하다.
- **스택/큐**: 스택은 LIFO(반복 전위 순회, 재귀 대체), 큐는 FIFO(레벨 순회, BFS)이다. 이번 실습에서 둘 다 직접 사용했다.
- **트리**: 전위·중위·후위는 DFS, 레벨 순회는 BFS이다. 높이는 간선 기준으로 빈 트리 -1, 리프 0이다.
- **BST**: 중위 순회가 오름차순이 되며 탐색·삽입·삭제는 O(h)이다. 삽입 순서에 따라 h가 log n에서 n까지 달라지므로 균형이 중요하다. 삭제는 자식 0개/1개/2개(후속자 또는 선행자)로 나누어 처리한다.